# Decoding search 2 on v9 adapter: repetition_penalty

In [ ]:
!pip uninstall -y -q torchao
!pip install -q transformers peft accelerate sacrebleu sentencepiece
import os, urllib.request, time
os.environ['CUDA_VISIBLE_DEVICES']='0'
import torch, sacrebleu
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

WORK='/kaggle/working'
RAW='https://raw.githubusercontent.com/SlavH/ashkarabar-to-grabar/main'
os.makedirs(f'{WORK}/data', exist_ok=True)
os.makedirs(f'{WORK}/adapter', exist_ok=True)
urllib.request.urlretrieve(f'{RAW}/data/test_real.tsv', f'{WORK}/data/test_real.tsv')
for f in ['adapter_config.json','adapter_model.safetensors']:
    urllib.request.urlretrieve(f'{RAW}/checkpoints/nllb_lora_grabar/{f}', f'{WORK}/adapter/{f}')

def read_pairs(path):
    out=[]
    with open(path,encoding='utf-8') as f:
        for line in f:
            if '\t' not in line: continue
            a,b=line.rstrip('\n').split('\t',1)
            if len(a.strip())>10 and len(b.strip())>10: out.append((a.strip(),b.strip()))
    return out

test_pairs=read_pairs(f'{WORK}/data/test_real.tsv')
print('test',len(test_pairs))

MODEL_NAME='facebook/nllb-200-distilled-600M'
SRC_LANG='hye_Armn'; TGT_LANG='hye_Armn'; MAXLEN=192
tok=AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
model=AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model=PeftModel.from_pretrained(model,f'{WORK}/adapter')
device='cuda' if torch.cuda.is_available() else 'cpu'
model.to(device).eval()
forced_bos=tok.convert_tokens_to_ids(TGT_LANG)

@torch.no_grad()
def translate(text, num_beams=6, length_penalty=1.2, no_repeat_ngram_size=4, repetition_penalty=1.0):
    tok.src_lang=SRC_LANG
    enc=tok(text,return_tensors='pt',truncation=True,max_length=MAXLEN).to(device)
    out=model.generate(**enc, forced_bos_token_id=forced_bos, max_new_tokens=MAXLEN,
                       num_beams=num_beams, no_repeat_ngram_size=no_repeat_ngram_size,
                       early_stopping=True, length_penalty=length_penalty,
                       repetition_penalty=repetition_penalty)
    return tok.batch_decode(out, skip_special_tokens=True)[0].strip()

best=None; results=[]
for rp in [1.0,1.05,1.1,1.15,1.2,1.3]:
  for nb in [6,5,4]:
    for lp in [1.2,1.1,1.0]:
      for nr in [4,3]:
        t0=time.time()
        preds=[translate(m,nb,lp,nr,rp) for m,_ in test_pairs]
        refs=[g for _,g in test_pairs]
        chrf=sacrebleu.corpus_chrf(preds,[refs]).score
        bleu=sacrebleu.corpus_bleu(preds,[refs]).score
        results.append((chrf,bleu,nb,lp,nr,rp))
        if chrf >= (best[0] if best else -1):
            best=(chrf,bleu,nb,lp,nr,rp)
        print(f'nb={nb} lp={lp} nr={nr} rp={rp}: chrF={chrf:.2f} BLEU={bleu:.2f} ({int(time.time()-t0)}s)')
results.sort(reverse=True)
print('TOP 10:')
for r in results[:10]: print(r)
chrf,bleu,nb,lp,nr,rp=best
print(f'BEST: beams={nb} lp={lp} nr={nr} rp={rp} chrF={chrf:.2f} BLEU={bleu:.2f}')
open(f'{WORK}/metrics_decode2.txt','w').write(f'chrF={chrf:.2f} BLEU={bleu:.2f} beams={nb} lp={lp} nr={nr} rp={rp}\n')